In [0]:
# Imports

from pyspark.sql.functions import to_timestamp, to_date, col, row_number, lit
from pyspark.sql.window import Window

In [0]:
# Identifica o que foi processado

try:
    ultimo_processado = spark.sql(
        "SELECT MAX(insert_dt) AS max_insert_dt FROM cambio_radar.silver.cotacoes_ptax"
    ).collect()[0]["max_insert_dt"]
except Exception:
    ultimo_processado = None  # tabela Silver ainda não existe (primeira execução)

filtro_data = ultimo_processado if ultimo_processado else "1900-01-01"

In [0]:
# Identifica o que é novo na camada bronze

df_novos = spark.sql(f"""
    SELECT *
    FROM cambio_radar.bronze.cotacoes_ptax
    WHERE insert_dt > '{filtro_data}'
""")

In [0]:
# Casts de tipo 

df_tratado = (
    df_novos
    .withColumn("dataHoraCotacao", to_timestamp("dataHoraCotacao"))
    .withColumn("insert_dt", to_timestamp("insert_dt"))
)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-6478425083530299>, line 4
      1 # Casts de tipo 
      3 df_tratado = (
----> 4     df_novos
      5     .withColumn("dataHoraCotacao", to_timestamp("dataHoraCotacao"))
      6     .withColumn("insert_dt", to_timestamp("insert_dt"))
      7 )

NameError: name 'df_novos' is not defined

In [0]:
# Valida nulos após os casts

nulos_apos_cast = df_tratado.filter(col("dataHoraCotacao").isNull()).count()

if nulos_apos_cast > 0:
    raise Exception(
        f"{nulos_apos_cast} registro(s) com dataHoraCotacao nulo após o cast — "
        "possível mudança no formato da fonte ou dado malformado na Bronze."
    )

In [0]:
# Aplica sequencia para boletins intermediarios

df_intermediario = df_tratado.filter(col("tipoBoletim") == "Intermediário")
df_outros = df_tratado.filter(col("tipoBoletim") != "Intermediário") \
    .withColumn("sequencia_intermediario", lit(None).cast("int"))

janela = Window.partitionBy("moeda", to_date("dataHoraCotacao")).orderBy("dataHoraCotacao")
df_intermediario = df_intermediario.withColumn("sequencia_intermediario", row_number().over(janela))

df_final = df_outros.unionByName(df_intermediario)
df_final.createOrReplaceTempView("novos_silver")

In [0]:
%sql

-- Grava os dados na camada Silver

MERGE INTO cambio_radar.silver.cotacoes_ptax AS destino
USING novos_silver AS origem
ON destino.moeda = origem.moeda
   AND destino.dataHoraCotacao = origem.dataHoraCotacao
WHEN NOT MATCHED THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1100,0,0,1100
